# 05 · 어떤 특징군과 평활 조건에 성능이 민감한가

Cat28에서 특징군 하나를 빼고 다시 학습한다. 6개 군 제거, 미리 정한 `robust` 특징 집합, 30초 평활 제거의 **8개 조건 × seed 3개 = 24개 후보**를 비교한다. 각 조건에서 검증 AP로 seed를 고른 뒤 `04의 Cat28 − 해당 조건`의 시험 성능 차이를 구한다. 같은 seed끼리의 차이도 별도로 남긴다.

**흐름:** 비교 조건 정의 → 두 특징 표·Cat28 예측 연결 → 학습·선택 → 같은 산모의 짝지은 차이 → seed별 방향 → 저장·해석

**입력:** `data/results/`의 `features.csv`, `features_unsmoothed.csv`, `protocol.json`, `pred_cat28.csv`, `seeds/pred_cat28_{42,43,44}.csv`  
**출력:** 같은 폴더의 05 후보·차이·상태 표와 조건별 예측 CSV

`03-요약된-파일/05-지표-기여와-평활.ipynb`의 특징군, 학습 설정, seed 선택과 산모 bootstrap을 유지한다. 코호트와 분할은 이 폴더의 00→01→02→04 결과를 따른다. 노트북을 위에서 아래로 실행하면 필요한 정의를 모두 볼 수 있다.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import average_precision_score

RESULTS = Path("data/results")
# Match 04's Cat28 budget and planned seeds for every alternative.
SEEDS = (42, 43, 44)
TREE_ITERATIONS = 1000
TREE_PATIENCE = 80
THREADS = 2
N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 42
ID_TYPES = {key: str for key in ("row_id", "record_id", "mother_id", "site")}
PREDICTION_COLUMNS = [
    "row_id", "record_id", "mother_id", "site", "target", "split",
    "record_all_normal", "signal_index", "score", "threshold",
]

### 출력에 남기는 오류 정보

오류는 고정된 분류와 건수로만 요약한다. `other_error`의 상세 원인은 내부 오류 파일에서 확인한다. 건수는 오류 내역 수이며, 같은 기록이 여러 번 포함될 수 있다.

In [ ]:
def issue_category(reason):
    # Match known phrases, but return ONLY fixed labels: never echo an exception.
    # Example: "FileNotFoundError: /private/DEMO.json" -> "missing_input".
    text = str(reason)
    groups = {
        "missing_input": ("FileNotFoundError", "No such file", "[Errno 2]"),
        "missing_dependency": ("No module named", "cannot import name", "ImportError"),
        "fhr_missing": ("FHR 결측률",),
        "invalid_wave": ("FHR과 TOCO", "측정 간격", "FHR은 0 이상", "신호 길이"),
        "invalid_bbox": ("Bbox",),
        "invalid_id": ("ID가", "산모 ID", "산모 목록", "행 정보 불일치", "구간 정보 불일치"),
        "invalid_labels": ("라벨이 올바르지", "라벨 수", "라벨·점수"),
        "no_usable_data": ("구간이 없습니다", "비었", "남은 산모가 없습니다"),
        "split_unavailable": ("분할", "양성·음성", "정상·비정상", "least populated class", "test_size"),
        "checkpoint_mismatch": ("체크포인트", "CNN 입력·설정"),
        "io_error": ("PermissionError", "Permission denied", "No space left", "OSError"),
        "invalid_value": ("ValueError", "TypeError", "유한", "could not convert", "ufunc"),
    }
    for category, phrases in groups.items():
        if any(phrase in text for phrase in phrases):
            return category
    return "other_error"

In [ ]:
def show_issue_counts(reasons):
    # Count issue entries, not unique people; one record may have several entries.
    categories = pd.Series([issue_category(reason) for reason in reasons], dtype="string")
    counts = categories.value_counts().rename_axis("category").to_frame("count")
    print("제외·오류 내역 수 =", len(categories))
    # Full messages, IDs, paths, and jobs stay in the internal CSV files.
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(counts)

## 1. 제거할 특징과 남길 특징을 먼저 정한다

특징군과 조건은 시험 점수를 보기 전에 고정한다. `minus_variability`는 `stv`와 `figo_baseline_var`만 빼고, 나머지 26개 특징을 사용한다. `robust`는 사건 9개와 수축 수·기저선·기저선 변이도 3개를 제외한 16개다. 이름이 강건성을 입증하지는 않는다.

In [ ]:
# Preserve the Cat28 feature order used by 02 and 04.
CAT28 = [
    "fhr_mean", "fhr_min", "fhr_max", "fhr_sd", "stv", "brady_frac", "tachy_frac",
    "n_decel", "decel_max_depth", "decel_time_frac", "n_accel",
    "toco_mean", "toco_max", "toco_sd", "n_contractions",
    "ft_corr0", "ft_corr_min", "ft_corr_min_lag_s", "figo_baseline", "figo_baseline_var",
    "n_early_decel", "n_late_decel", "n_variable_decel",
    "n_severe_decel", "n_prolonged_decel", "hist_width", "hist_median", "hist_mode",
]
GROUPS = {
    # Four FHR level/spread features: mean, minimum, maximum, standard deviation.
    "signal": CAT28[:4],
    "baseline": ["figo_baseline", "hist_median", "hist_mode", "hist_width"],
    "variability": ["stv", "figo_baseline_var"],
    "range": ["brady_frac", "tachy_frac"],
    # Nine event features are removed together, not one at a time.
    "events": CAT28[7:11] + CAT28[20:25],
    # Seven TOCO and FHR–TOCO coupling features -> 21 retained inputs.
    "contractions_coupling": CAT28[11:18],
}
# Example: events contains 4 basic event features + 5 deceleration types.
# n_severe_decel stays in its declared group, although it is always 0 in 5 minutes.

In [ ]:
arms = {}
for group, removed_columns in GROUPS.items():
    # Example: remove [stv, figo_baseline_var] -> 28 - 2 = 26 inputs.
    # Keep the remaining columns in Cat28 order.
    arms["minus_" + group] = [column for column in CAT28 if column not in removed_columns]

robust_exclusions = GROUPS["events"] + [
    "n_contractions", "figo_baseline", "figo_baseline_var",
]
# Example: 28 - 9 event features - 3 other features = 16 robust inputs.
arms["robust"] = [column for column in CAT28 if column not in robust_exclusions]
# Unsmoothed changes feature values, while keeping all 28 feature names.
arms["unsmoothed"] = CAT28

In [ ]:
# One row describes one comparison arm, not one fitted seed.
arm_rows = []
for arm, columns in arms.items():
    arm_rows.append(dict(
        arm=arm,
        n_features=len(columns),
        removed=", ".join(column for column in CAT28 if column not in columns),
        source="features_unsmoothed.csv" if arm == "unsmoothed" else "features.csv",
    ))
arm_table = pd.DataFrame(arm_rows)
display(arm_table)
print("minus_variability removes =", GROUPS["variability"])
print("robust keeps =", arms["robust"])
print("planned candidates =", len(arms) * len(SEEDS))

## 2. 같은 구간과 같은 산모 분할을 연결한다

`features.csv`는 30초 평활, `features_unsmoothed.csv`는 평활 평균 계산만 생략한 특징이다. **두 조건 모두 02에서 결측을 보간한 결과**다. 여기서 원파형을 다시 가공하지 않는다.

두 특징 표는 같은 구간 ID·산모·기록·라벨·분할을 가져야 한다. 행 순서가 다르면 ID로 정렬하고, 구간이 빠졌으면 비교를 보류한다. 04의 Cat28 예측도 현재 검증·시험 구간에 맞춰 확인한다.

In [ ]:
def align_rows(left, right):
    # Example: left IDs [A, B], right IDs [B, A] -> reorder right to [A, B].
    # Missing B is an error; comparing only the intersection changes the cohort.
    if left["row_id"].duplicated().any() or right["row_id"].duplicated().any():
        raise ValueError("구간 ID가 중복됩니다")
    if set(left["row_id"]) != set(right["row_id"]):
        raise ValueError("비교할 구간 집합이 다릅니다")
    # Reordering rows never changes their labels, records, or mother IDs.
    aligned = right.set_index("row_id").loc[left["row_id"]].reset_index()
    for column in ("mother_id", "target", "record_id"):
        if not np.array_equal(left[column].to_numpy(), aligned[column].to_numpy()):
            raise ValueError("구간 정보 불일치: " + column)
    return aligned

In [ ]:
def split_holdout(frame):
    parts = []
    for split in ("train", "val", "test"):
        # This reads saved assignments; it does not draw a fresh split per arm.
        part = frame.loc[frame["split"].eq(split)].copy()
        # Example: [0, 0, 0] alone cannot support the planned binary comparison.
        if set(part["target"].unique()) != {0, 1}:
            raise ValueError("train/val/test 각각 정상·비정상 라벨이 필요합니다")
        parts.append(part)
    return parts

In [ ]:
feature_table = pd.DataFrame()
unsmoothed_table = pd.DataFrame()
baseline_prediction = pd.DataFrame()
protocol = {}
input_error = None
issues = []

try:
    # IDs are strings; "00DEMO527183" is a fictional example with leading zeros.
    feature_table = pd.read_csv(RESULTS / "features.csv", dtype=ID_TYPES)
    # Both tables contain interpolated signals' features; only averaging differs.
    unsmoothed_table = pd.read_csv(RESULTS / "features_unsmoothed.csv", dtype=ID_TYPES)
    protocol = json.loads((RESULTS / "protocol.json").read_text(encoding="utf-8"))
    baseline_prediction = pd.read_csv(RESULTS / "pred_cat28.csv", dtype=ID_TYPES)
except (OSError, ValueError) as error:
    input_error = str(error)

In [ ]:
ready = False
training = pd.DataFrame()
validation = pd.DataFrame()
testing = pd.DataFrame()
if input_error is None:
    try:
        if feature_table.empty or not protocol.get("split_ready", False):
            raise ValueError("특징 표가 비었거나 산모 분할이 준비되지 않았습니다")
        # Example: a missing stv column stops the complete planned comparison.
        feature_table[CAT28]
        unsmoothed_table[CAT28]
        # Example: mother A cannot contribute some train rows and some test rows.
        if feature_table.groupby("mother_id")["split"].nunique().max() > 1:
            raise ValueError("같은 산모가 여러 분할에 배정되어 있습니다")
        if set(feature_table["split"]) != {"train", "val", "test"}:
            raise ValueError("train/val/test 분할이 올바르지 않습니다")
        unsmoothed_table = align_rows(feature_table, unsmoothed_table)
        # signal_index must still point to the same prepared waveform.
        for column in ("split", "signal_index"):
            if not np.array_equal(feature_table[column], unsmoothed_table[column]):
                raise ValueError("두 전처리의 구간 정보 불일치: " + column)
        training, validation, testing = split_holdout(feature_table)
        ready = True
    except (KeyError, ValueError) as error:
        input_error = str(error)
if input_error is not None:
    issues.append(dict(stage="input", job="cohort", reason=input_error))

In [ ]:
def check_reference(frame, prediction):
    # Both 04 and 05 store val + test, with one probability per segment.
    aligned = align_rows(frame, prediction)
    if not np.array_equal(frame["split"], aligned["split"]):
        raise ValueError("Cat28 예측의 검증·시험 배정이 현재 입력과 다릅니다")
    if not np.isfinite(aligned["score"].to_numpy(dtype=float)).all():
        raise ValueError("Cat28 예측에 비유한 점수가 있습니다")
    return aligned

In [ ]:
baseline_test = pd.DataFrame()
baseline_by_seed = {}
if ready:
    evaluation_pool = pd.concat([validation, testing])
    try:
        baseline_prediction = check_reference(evaluation_pool, baseline_prediction)
        # Example: 10 validation + 5 test rows -> only 5 baseline evaluation rows.
        baseline_test = baseline_prediction.loc[baseline_prediction["split"].eq("test")].copy()
    except (KeyError, ValueError) as error:
        ready = False
        input_error = str(error)
        issues.append(dict(stage="input", job="selected_cat28", reason=input_error))

if ready:
    for seed in SEEDS:
        try:
            reference = pd.read_csv(RESULTS / "seeds" / f"pred_cat28_{seed}.csv", dtype=ID_TYPES)
            baseline_by_seed[seed] = check_reference(evaluation_pool, reference)
        except (OSError, KeyError, ValueError) as error:
            # A missing seed reference affects that same-seed comparison only.
            issues.append(dict(stage="reference", job=f"cat28:{seed}", reason=str(error)))

In [ ]:
print("ready =", ready)
if ready:
    # Windows are the model rows; mothers are the independent split units.
    display(feature_table.groupby("split").agg(
        mothers=("mother_id", "nunique"), windows=("row_id", "size"),
        abnormal_fraction=("target", "mean"),
    ))
    print("Cat28 test rows =", len(baseline_test))
    print("available Cat28 seeds =", list(baseline_by_seed))
    # Inspect exactly the same segment under both preprocessing conditions.

else:
    print('05 미산출: 01~04의 입력을 확인하세요.', '[details omitted from notebook output]')

## 3. 조건 하나·seed 하나를 학습한다

특징군 제거와 robust 조건은 평활 표에서 입력 열만 바꾼다. `unsmoothed`는 28개 열을 모두 유지하고 비평활 표로 학습·검증·시험한다. 모든 조건은 CatBoost depth 6, 학습률 0.05, 최대 1,000회, 검증 조기중단 80회를 공유한다. 시험 집합은 조기중단이나 seed 선택에 쓰지 않는다.

In [ ]:
def fit_catboost(train, tune, columns, seed):
    from catboost import CatBoostClassifier

    # Every arm uses the same CatBoost settings as 04; inputs are specified below.
    model = CatBoostClassifier(
        iterations=TREE_ITERATIONS, depth=6, learning_rate=0.05,
        loss_function="Logloss", eval_metric="PRAUC", auto_class_weights="SqrtBalanced",
        random_seed=seed, thread_count=THREADS,
        allow_writing_files=False, verbose=False,
    )
    # Only tune controls early stopping; test is never passed to fit().
    model.fit(
        train[columns], train["target"],
        eval_set=(tune[columns], tune["target"]),
        # Example: 80 unimproved rounds can stop before the 1,000-round cap.
        early_stopping_rounds=TREE_PATIENCE,
    )
    return model

In [ ]:
def make_prediction(frame, scores):
    # Example: row_id="DEMO221841_0:0", score=0.7 means P(abnormal)=0.7.
    prediction = frame[PREDICTION_COLUMNS[:-2]].copy()
    # Scores are row-aligned: scores[i] belongs to frame.iloc[i].
    prediction["score"] = np.asarray(scores, dtype=float)
    prediction["threshold"] = np.nan  # No alarm threshold is selected here.
    if prediction["row_id"].duplicated().any():
        raise ValueError("예측 구간 ID가 중복됩니다")
    if not np.isfinite(prediction["score"]).all():
        raise ValueError("예측 확률에 비유한 값이 있습니다")
    return prediction

In [ ]:
def fit_candidate(data, columns, seed):
    train, tune, test = split_holdout(data)
    # Example: minus_variability fits a (training windows, 26) input table.
    model = fit_catboost(train, tune, columns, seed)
    # Store val and test together so later notebooks can reuse the row-level scores.
    evaluation_pool = pd.concat([tune, test])
    # Column 1 is P(abnormal); column 0 would be P(normal).
    scores = model.predict_proba(evaluation_pool[columns])[:, 1]
    prediction = make_prediction(evaluation_pool, scores)

    # Only validation AP participates in seed selection.
    validation_rows = prediction.loc[prediction["split"].eq("val")]
    val_ap = average_precision_score(validation_rows["target"], validation_rows["score"])
    # Return the whole prediction table and one validation selection score.
    return prediction, float(val_ap)

In [ ]:
ranking_rows = []
seed_predictions = {}
if ready:
    for arm, columns in arms.items():
        # Example: minus_signal reads 24 smooth features; unsmoothed reads 28 unaveraged features.
        data = unsmoothed_table if arm == "unsmoothed" else feature_table
        for seed in SEEDS:
            try:
                prediction, val_ap = fit_candidate(data, columns, seed)
                # Example key: ("minus_variability", 43); the two items identify one fit.
                seed_predictions[(arm, seed)] = prediction
                ranking_rows.append(dict(arm=arm, seed=seed, val_AP=val_ap))
            except Exception as error:
                # Record one failed fit and continue with the other planned seeds.
                issues.append(dict(stage="fit", job=f"{arm}:{seed}", reason=str(error)))
        completed = sum(row["arm"] == arm for row in ranking_rows)
        print(arm, "completed seeds =", completed, "/", len(SEEDS), flush=True)

In [ ]:
# One row = one successfully fitted arm/seed, in planned arm and seed order.
selection_table = pd.DataFrame(ranking_rows, columns=["arm", "seed", "val_AP"])
print("successful candidates =", len(selection_table), "/", len(arms) * len(SEEDS))
display(selection_table)

## 4. 검증 AP로 조건별 seed 하나를 고정한다

예정된 세 seed가 모두 성공한 조건에서만 하나를 고른다. 검증 AP가 같으면 작은 seed를 선택한다. 선택 후 재학습하지 않고 이미 만든 예측을 사용한다. 기준 Cat28의 seed는 04에서 선택한 그대로다. 두 조건의 선택 seed가 서로 다를 수 있으므로 뒤에서 같은 seed끼리도 비교한다.

In [ ]:
def choose_seed(candidates):
    # Example: successful seeds [42, 44] are incomplete; do not choose from two.
    if len(candidates) != len(SEEDS) or set(candidates["seed"]) != set(SEEDS):
        raise ValueError("동일 seed 집합 미완료: 주 비교 미산출")
    # Example: AP 0.52 at seeds 43 and 44 -> seed 43 wins the tie.
    best = max(candidates.itertuples(), key=lambda row: (row.val_AP, -row.seed))
    # Selection is independent of the Cat28 seed already chosen in 04.
    return int(best.seed), float(best.val_AP)

In [ ]:
# Check the tie rule with a tiny table before applying it to the real candidates.
example_candidates = pd.DataFrame({"seed": [42, 43, 44], "val_AP": [0.50, 0.52, 0.52]})
# Limit the example to the planned seeds if the experiment settings were reduced.
example_candidates = example_candidates.loc[example_candidates["seed"].isin(SEEDS)]
example_seed, example_ap = choose_seed(example_candidates)
print("example selected seed =", example_seed)
print("example validation AP =", example_ap)

In [ ]:
selected = {}
selected_predictions = {}
if ready:
    for arm in arms:
        candidates = selection_table.loc[selection_table["arm"].eq(arm)]
        try:
            seed, val_ap = choose_seed(candidates)
            selected[arm] = dict(seed=seed, val_AP=val_ap)
            # Reuse probabilities from the selected fit; do not retrain on train + val.
            selected_predictions[arm] = seed_predictions[(arm, seed)]
        except ValueError as error:
            issues.append(dict(stage="selection", job=arm, reason=str(error)))

# This table contains validation decisions, not a ranking of test performance.
print("selected arms =", len(selected), "/", len(arms))
display(pd.DataFrame.from_dict(selected, orient="index"))

In [ ]:
# Report only the number of prediction rows per selected comparison arm.
for arm, prediction in selected_predictions.items():
    print(arm, "prediction rows =", len(prediction))

## 5. 같은 시험 산모를 함께 뽑아 성능 차이를 계산한다

차이의 방향은 항상 **Cat28 − 비교 조건**이다. AP·AUROC 차이가 양수면 제거·평활 변경 후 성능이 낮아진 것이다. Brier는 작을수록 좋으므로 음수일 때 Cat28의 오차가 더 작다.

산모 한 명을 두 번 뽑으면 그 산모의 모든 구간을 두 번 센다. 양쪽 모델에 같은 횟수를 적용한 뒤 점수를 빼야 짝지은 비교가 된다. 2,000회 중 양성·음성이 모두 있는 반복만 사용하며, 유효한 차이의 2.5%·97.5% 분위수를 저장한다.

In [ ]:
def prepare_metrics(prediction):
    labels = prediction["target"].to_numpy(dtype=int)
    scores = prediction["score"].to_numpy(dtype=float)
    if not len(prediction) or set(labels) - {0, 1} or not np.isfinite(scores).all():
        raise ValueError("평가 라벨·점수가 올바르지 않거나 평가 집합이 비었습니다")

    # Example: mother IDs [A, A, B] -> mother_index [0, 0, 1].
    mothers, mother_index = np.unique(prediction["mother_id"], return_inverse=True)
    # Example: scores [0.2, 0.9, 0.5] -> descending row order [1, 2, 0].
    order = np.argsort(-scores, kind="stable")
    # Example: sorted scores [0.9, 0.9, 0.3] -> group ends [1, 2].
    # Tied scores are one threshold, independent of their row order.
    ends = np.r_[np.flatnonzero(np.diff(scores[order])), len(prediction) - 1]
    return dict(
        labels=labels, scores=scores, mothers=mothers,
        mother_index=mother_index, order=order, ends=ends,
    )

In [ ]:
def weighted_metrics(data, counts=None):
    if counts is None:
        # Point estimate: every mother appears once, so every window has weight 1.
        counts = np.ones(len(data["mothers"]))
    # Example: mother counts [2, 0] -> weights [2, 2, 0] for [A, A, B].
    weights = counts[data["mother_index"]]
    labels = data["labels"]
    # Denominators count weighted abnormal and normal *windows*.
    positive = np.sum(weights * labels)
    negative = np.sum(weights * (1 - labels))
    # Skip a resample containing only normal or only abnormal windows.
    if not positive or not negative:
        return None

    order, ends = data["order"], data["ends"]
    # Count windows at or above each distinct score threshold.
    # Example: distinct descending scores with labels [1, 0, 1] -> TP [0, 1, 1, 2].
    true_positive = np.r_[0, np.cumsum((weights * labels)[order])[ends]]
    false_positive = np.r_[0, np.cumsum((weights * (1 - labels))[order])[ends]]
    recall = true_positive / positive
    false_positive_rate = false_positive / negative
    predicted_positive = np.maximum(true_positive[1:] + false_positive[1:], 1e-30)
    # Example: descending labels [1, 0, 1, 0] -> AP = (1 + 2/3)/2 = 5/6.
    ap = np.sum(np.diff(recall) * true_positive[1:] / predicted_positive)
    # Those labels with distinct scores give AUROC=3/4.
    auc = np.sum(np.diff(false_positive_rate) * (recall[1:] + recall[:-1]) / 2)
    # Example: label 1, score 0.9 contributes (0.9 - 1)^2 = 0.01.
    brier = np.sum(weights * (data["scores"] - labels) ** 2) / weights.sum()
    return dict(AP=ap, AUROC=auc, Brier=brier)

In [ ]:
def resample_differences(base_data, alternative_data, n_boot, seed):
    samples = {metric: [] for metric in ("AP", "AUROC", "Brier")}
    mother_count = len(base_data["mothers"])
    rng = np.random.default_rng(seed)
    for _ in range(n_boot):
        # Example: draw [0, 0, 2] from 3 mothers -> counts [2, 0, 1].
        draws = rng.integers(mother_count, size=mother_count)
        counts = np.bincount(draws, minlength=mother_count)
        # Mother A drawn twice repeats all her records and both twins' windows.
        # Both sides use identical counts, not independent resamples.
        base_result = weighted_metrics(base_data, counts)
        alternative_result = weighted_metrics(alternative_data, counts)
        if base_result is None or alternative_result is None:
            continue
        for metric in samples:
            # Example: resampled AP 0.60 - 0.55 -> difference 0.05.
            difference = base_result[metric] - alternative_result[metric]
            if np.isfinite(difference):
                samples[metric].append(difference)
    return samples

In [ ]:
def paired_intervals(base, alternative, n_boot=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    # Align rows first so mother counts refer to the same patients on both sides.
    alternative = align_rows(base, alternative)
    base_data = prepare_metrics(base)
    alternative_data = prepare_metrics(alternative)
    base_point = weighted_metrics(base_data)
    alternative_point = weighted_metrics(alternative_data)
    if base_point is None or alternative_point is None:
        raise ValueError("평가에 정상·비정상 구간이 모두 필요합니다")
    samples = resample_differences(base_data, alternative_data, n_boot, seed)

    rows = []
    for metric in base_point:
        low, high = np.nan, np.nan
        if samples[metric]:
            # Example: an AP interval [-0.01, 0.04] includes zero difference.
            low, high = np.quantile(samples[metric], [0.025, 0.975])
        # n_boot=0 yields a point difference with NaN interval endpoints.
        rows.append(dict(
            metric=metric, point=base_point[metric] - alternative_point[metric],
            # Valid draws can be fewer than 2,000 when one-class draws are skipped.
            low=low, high=high, valid_bootstrap=len(samples[metric]),
            mothers=len(base_data["mothers"]), segments=len(base),
        ))
    return pd.DataFrame(rows)

In [ ]:
# Two mothers, two windows each: both contain labels 1 and 0.
example_base = pd.DataFrame({
    "row_id": ["a", "b", "c", "d"], "record_id": ["r1", "r1", "r2", "r2"],
    "mother_id": ["A", "A", "B", "B"], "target": [1, 0, 1, 0],
    "score": [0.9, 0.8, 0.7, 0.1],
})
example_alternative = example_base.copy()
example_alternative["score"] = [0.8, 0.9, 0.7, 0.1]
# Base AP=5/6, alternative AP=7/12 -> AP difference=1/4.
# Base Brier=0.1875, alternative Brier=0.2375 -> difference=-0.05.
display(paired_intervals(example_base, example_alternative, n_boot=0))
# An identical model must have zero point difference and zero interval width.
display(paired_intervals(example_base, example_base, n_boot=20))

In [ ]:
paired_rows = []
for arm, prediction in selected_predictions.items():
    # Validation chose the seed; only test windows enter these intervals.
    alternative_test = prediction.loc[prediction["split"].eq("test")]
    try:
        # Keep Cat28 on the left: AP 0.60 - 0.55 means a +0.05 removal effect.
        delta = paired_intervals(baseline_test, alternative_test)
        # Each completed arm contributes three rows: AP, AUROC, and Brier.
        paired_rows.append(delta.assign(
            comparison="cat28 - " + arm, selected_seed=selected[arm]["seed"],
        ))
    except (KeyError, ValueError) as error:
        issues.append(dict(stage="paired", job=arm, reason=str(error)))

In [ ]:
DELTA_COLUMNS = ["metric", "point", "low", "high", "valid_bootstrap", "mothers", "segments"]
paired_table = pd.DataFrame(columns=DELTA_COLUMNS + ["comparison", "selected_seed"])
if paired_rows:
    paired_table = pd.concat(paired_rows, ignore_index=True)
# Example: 8 completed comparisons × 3 metrics = 24 rows.
print("completed paired comparisons =", len(paired_rows), "/", len(arms))
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(paired_table)


In [ ]:
ap_comparisons = paired_table.loc[paired_table["metric"].eq("AP")]
if ap_comparisons.empty:
    print("AP 차이를 그릴 비교 결과가 없습니다.")
else:
    positions = np.arange(len(ap_comparisons))
    figure, axis = plt.subplots(figsize=(8, max(3, len(ap_comparisons) * 0.4)))
    # Each line spans one comparison's 95% mother-bootstrap interval.
    axis.hlines(positions, ap_comparisons["low"], ap_comparisons["high"], color="C0")
    # Dots to the right of zero favor Cat28 for AP.
    axis.scatter(ap_comparisons["point"], positions)
    axis.axvline(0, color="gray", linestyle="--")
    axis.set_yticks(positions, ap_comparisons["comparison"])
    axis.set_xlabel("AP difference (Cat28 minus alternative)")
    axis.set_title("Group removal and smoothing: Cat28 minus alternative")
    figure.tight_layout()
    plt.show()

## 6. 같은 seed끼리도 방향을 확인한다

주 비교는 각 조건에서 선택된 seed를 사용한다. 여기서는 `Cat28 seed 42 − 비교 조건 seed 42`처럼 seed를 맞춘다. 새로운 모델을 학습하거나 다시 선택하지 않는다. 이 표는 점 차이만 저장하므로 `valid_bootstrap=0`, `low/high=NaN`이다. 기준 seed 예측이 없으면 해당 차이만 보류하고 사유를 기록한다.

In [ ]:
same_seed_rows = []
for (arm, seed), prediction in seed_predictions.items():
    # Missing Cat28 seed 43 does not invalidate available seed 42/44 comparisons.
    if seed not in baseline_by_seed:
        continue
    reference = baseline_by_seed[seed]
    base_test = reference.loc[reference["split"].eq("test")]
    alternative_test = prediction.loc[prediction["split"].eq("test")]
    try:
        # Same seed on both sides; no seed selection or bootstrap here.
        delta = paired_intervals(base_test, alternative_test, n_boot=0)
        same_seed_rows.append(delta.assign(seed=seed, comparison="cat28 - " + arm))
    except (KeyError, ValueError) as error:
        issues.append(dict(stage="same_seed", job=f"{arm}:{seed}", reason=str(error)))

In [ ]:
same_seed_table = pd.DataFrame(columns=DELTA_COLUMNS + ["seed", "comparison"])
if same_seed_rows:
    same_seed_table = pd.concat(same_seed_rows, ignore_index=True)
    ap_rows = same_seed_table.loc[same_seed_table["metric"].eq("AP")]
    # Example: one row for minus_signal, one column per seed 42/43/44.
    # A missing comparison remains NaN; never fill it with a zero difference.
    ap_by_seed = ap_rows.pivot(index="comparison", columns="seed", values="point")
    display(ap_by_seed)
    ap_by_seed.plot.barh(figsize=(9, 5), title="Same-seed AP differences")
    plt.axvline(0, color="gray")
    plt.tight_layout()
    plt.show()
else:
    print("동일 seed 비교 결과가 없습니다.")

In [ ]:
# Every row is one model comparison, one seed, and one metric (AP/AUROC/Brier).
# These are point differences; same-seed comparisons have no bootstrap CI.
summary_columns = ["comparison", "seed", "metric", "point", "mothers", "segments"]
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(same_seed_table[summary_columns])

## 7. 이번 실행의 후보·예측·차이를 저장한다

| 파일 | 한 행 또는 파일의 의미 |
| --- | --- |
| `05_selection.csv` | 조건·seed 하나의 검증 AP |
| `05_paired.csv` | 선택된 Cat28과 선택된 비교 조건의 시험 지표 차이·95% CI |
| `05_same_seed.csv` | 동일 seed끼리의 시험 지표 점 차이 |
| `seeds/pred_{arm}_{seed}.csv` | 성공한 후보의 검증·시험 구간별 확률 |
| `pred_{arm}.csv` | 해당 조건에서 선택한 seed의 검증·시험 예측 |
| `05_issues.csv` | 입력·학습·선택·비교에서 보류한 작업과 이유 |
| `05_status.json` | 완료 상태, 조건별 선택 seed, 완료 비교 수와 설정 |

원본의 예측·결과 파일명과 열 순서를 유지한다. 예정된 결과 경로를 빈 스키마로도 갱신해 이전 실행의 성공 결과가 남지 않게 한다. 04의 Cat28 예측은 입력으로 읽으며, 이 노트북의 비교 조건 이름만 저장한다.

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "seeds").mkdir(exist_ok=True)
# Empty output tables retain their columns so an unavailable rerun is explicit.
selection_table.to_csv(RESULTS / "05_selection.csv", index=False)
paired_table.to_csv(RESULTS / "05_paired.csv", index=False)
same_seed_table.to_csv(RESULTS / "05_same_seed.csv", index=False)

for arm in arms:
    for seed in SEEDS:
        # Example: a failed minus_signal seed 43 gets a header-only CSV.
        prediction = seed_predictions.get((arm, seed), pd.DataFrame(columns=PREDICTION_COLUMNS))
        prediction.to_csv(RESULTS / "seeds" / f"pred_{arm}_{seed}.csv", index=False)
    prediction = selected_predictions.get(arm, pd.DataFrame(columns=PREDICTION_COLUMNS))
    prediction.to_csv(RESULTS / f"pred_{arm}.csv", index=False)

In [ ]:
issue_table = pd.DataFrame(issues, columns=["stage", "job", "reason"])
issue_table.to_csv(RESULTS / "05_issues.csv", index=False)
all_pairs_complete = len(paired_rows) == len(arms)
all_seeds_compared = len(same_seed_rows) == len(arms) * len(SEEDS)
if not ready:
    status_name = "unavailable"
elif all_pairs_complete and all_seeds_compared and issue_table.empty:
    status_name = "complete"
else:
    status_name = "partial"

# completed_arms counts successful paired comparisons, not individual fits.
status = dict(
    status=status_name, completed_arms=len(paired_rows), expected_arms=len(arms),
    selected_arms=selected, candidates_completed=len(selection_table),
    candidates_expected=len(arms) * len(SEEDS), same_seed_comparisons=len(same_seed_rows),
    input_error=input_error, seeds=list(SEEDS), tree_iterations=TREE_ITERATIONS,
    tree_patience=TREE_PATIENCE, threads=THREADS,
    bootstrap=N_BOOTSTRAP, bootstrap_seed=BOOTSTRAP_SEED,
)
(RESULTS / "05_status.json").write_text(
    json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8",
)
print("status =", status_name)
print("paired comparisons =", len(paired_rows), "/", len(arms))
print("same-seed comparisons =", len(same_seed_rows), "/", len(arms) * len(SEEDS))
print("issues =", len(issue_table))
if not issue_table.empty:
    print("issue count =", len(issue_table))

## 제거 효과를 해석하는 범위

상관된 특징들은 서로를 대신할 수 있다. 군 제거 후 성능 차이는 다른 특징이 함께 있는 상태에서 모델을 다시 학습했을 때의 예측 변화다. 생리적 원인이나 전문의의 판단 과정을 입증하지 않는다. 0을 포함하는 CI만으로 기여가 없다고 결론내리지 않으며, 여러 특징군의 비교는 탐색적으로 읽는다.

평활 제거 효과는 같은 구간의 측정·학습 조건 변경에 대한 민감도다. 두 조건 모두 보간을 거쳤고, `n_severe_decel`은 5분보다 긴 사건을 이 구간 안에서 셀 수 없어 구조적으로 0이다. 이 제약을 특징의 임상적 무용성으로 해석하지 않는다. 95% CI는 고정된 선택 모델의 시험 산모를 재표집하며, 재학습과 seed 선택 전체의 불확실성을 포함하지 않는다.

### 반출용 제외·오류 집계

개별 ID·경로·오류 전문을 제외한 분류별 건수다.

In [ ]:
show_issue_counts(issue_table["reason"])